# DATA EXTRACTION FROM MIMIC-III FOR MetaLearning4EHRs

This file is for extracting heart-related diseases informations(ICD9 Code start with 42)

In [ ]:
# Import libraries
import pandas as pd
import psycopg2
import os
from pathlib import Path

In [ ]:
# Update connection details to MIMIC-III
conn = psycopg2.connect("dbname = postgres user= postgres password= postgres host=localhost port=5432")

# Update the path for data extraction here
export_dir=r"data\MIMICIII_last48h"
Path(export_dir).mkdir(exist_ok=True, parents=True)

## Extract considered icd9 code

icd9_code start with V and E are not considered

In [ ]:
query = """
SELECT d.*, d_icd.long_title 
FROM (
  SELECT icd9_code, count(DISTINCT hadm_id) as count
  FROM diagnoses_icd d, patients p
  WHERE seq_num=1 AND p.subject_id=d.subject_id
  group by d.icd9_code
) d, d_icd_diagnoses d_icd
WHERE d.icd9_code=d_icd.icd9_code
  AND d.icd9_code NOT LIKE 'V%'
  AND d.icd9_code NOT LIKE 'E%'
order by count desc
"""
df = pd.read_sql_query(query,conn, dtype={'icd9_code': str})
df.to_csv(os.path.join(export_dir, "icd9.csv"),index=False, sep=',')
df.info()

## Demography

Age, Gender, ethnicity<br>
16 < Age < 90, in_hospital_expire_flag = 0

In [ ]:
query = f"""
SELECT *
FROM (SELECT distinct 
	  ad.subject_id,
      ad.hadm_id, 
	  ad.ethnicity, 
	  p.gender,
	  d.icd9_code,
	  EXTRACT(EPOCH FROM (ad.dischtime-ad.admittime))/3600 as staytime, 
	  EXTRACT(EPOCH FROM (ad.admittime-p.dob))/(3600*24*365) as age
	FROM admissions ad, patients p, diagnoses_icd d
	WHERE ad.subject_id = p.subject_id
        AND ad.hadm_id = d.hadm_id
	    AND AGE(COALESCE(p.dod_hosp, 'infinity'::timestamp), ad.admittime) > INTERVAL '2 days'
	    AND d.seq_num = 1
	    AND d.icd9_code NOT LIKE 'V%'
	    AND d.icd9_code NOT LIKE 'E%'
	    AND ad.hospital_expire_flag = 0
	  )
WHERE age > 16 AND age < 90
AND staytime >= 48;
"""
df = pd.read_sql_query(query, conn)
df.to_csv(os.path.join(export_dir, "demographics.csv"),index=False,sep=',')
df.info()

considered_hadm_id = df['hadm_id'].tolist()
considered_hadm_id_str = ', '.join(map(str, considered_hadm_id))

## Get Labels
Labels are 30-day readmission, and 90-day mortality

In [ ]:
# extract the 30-day readmission label
query = f"""
SELECT *
FROM (
    SELECT 
        ad0.row_id,
        ad0.subject_id, 
        ad0.hadm_id, 
        CASE 
            WHEN MIN(EXTRACT(EPOCH FROM ad1.admittime - ad0.dischtime)/(3600*24)) <= 30 THEN 1 
            ELSE 0 
        END AS days_30_readmission_flag
    FROM 
        admissions ad0
    LEFT JOIN 
        admissions ad1 ON ad0.subject_id = ad1.subject_id AND ad0.row_id < ad1.row_id
    GROUP BY 
        ad0.row_id, ad0.subject_id, ad0.hadm_id) as g
WHERE 
    g.hadm_id in ({considered_hadm_id_str});
"""

df = pd.read_sql_query(query, conn)
df.to_csv(os.path.join(export_dir, "readmission.csv"),index=False,sep=',')
df.info()

In [ ]:
# extract the 90-day mortality label
query = f"""
SELECT *
FROM (
    SELECT 
        ad.subject_id, 
        ad.hadm_id, 
        CASE 
            WHEN EXTRACT(EPOCH FROM p.dod - ad.dischtime)/(3600*24) <= 90 THEN 1 
            ELSE 0 
        END AS days_90_expire_flag
    FROM 
        patients p, 
        admissions ad
    WHERE 
        p.subject_id = ad.subject_id
        AND ad.hospital_expire_flag = 0
        ) as g
WHERE
    g.hadm_id in ({considered_hadm_id_str});
"""

df = pd.read_sql_query(query, conn)
df.to_csv(os.path.join(export_dir, "mortality.csv"),index=False,sep=',')
df.info()

## Vital Signs

In [ ]:
query = f"""
-- This query pivots the vital signs for a patient with stay time > 48 hours
-- Vital signs include heart rate, blood pressure, respiration rate, SPO2, glucose, and temperature

SELECT pvt.subject_id, pvt.hadm_id, pvt.charttime
, min(case when VitalID = 1 then valuenum ELSE NULL END) AS heartrate_min
, max(case when VitalID = 1 then valuenum ELSE NULL END) AS heartrate_max
, avg(case when VitalID = 1 then valuenum ELSE NULL END) AS heartrate_mean
, min(case when VitalID = 2 then valuenum ELSE NULL END) AS sysbp_min
, max(case when VitalID = 2 then valuenum ELSE NULL END) AS sysbp_max
, avg(case when VitalID = 2 then valuenum ELSE NULL END) AS sysbp_mean
, min(case when VitalID = 3 then valuenum ELSE NULL END) AS diasbp_min
, max(case when VitalID = 3 then valuenum ELSE NULL END) AS diasbp_max
, avg(case when VitalID = 3 then valuenum ELSE NULL END) AS diasbp_mean
, min(case when VitalID = 4 then valuenum ELSE NULL END) AS meanbp_min
, max(case when VitalID = 4 then valuenum ELSE NULL END) AS meanbp_max
, avg(case when VitalID = 4 then valuenum ELSE NULL END) AS meanbp_mean
, min(case when VitalID = 5 then valuenum ELSE NULL END) AS resprate_min
, max(case when VitalID = 5 then valuenum ELSE NULL END) AS resprate_max
, avg(case when VitalID = 5 then valuenum ELSE NULL END) AS resprate_mean
, min(case when VitalID = 6 then valuenum ELSE NULL END) AS tempc_min
, max(case when VitalID = 6 then valuenum ELSE NULL END) AS tempc_max
, avg(case when VitalID = 6 then valuenum ELSE NULL END) AS tempc_mean
, min(case when VitalID = 7 then valuenum ELSE NULL END) AS spo2_min
, max(case when VitalID = 7 then valuenum ELSE NULL END) AS spo2_max
, avg(case when VitalID = 7 then valuenum ELSE NULL END) AS spo2_mean
, min(case when VitalID = 8 then valuenum ELSE NULL END) AS glucose_min
, max(case when VitalID = 8 then valuenum ELSE NULL END) AS glucose_max
, avg(case when VitalID = 8 then valuenum ELSE NULL END) AS glucose_mean

FROM  (
  select ad.subject_id, ad.hadm_id, EXTRACT(EPOCH FROM ce.charttime - ad.admittime) as charttime
  , case
    when itemid in (211,220045) and valuenum > 0 and valuenum < 300 then 1 -- HeartRate
    when itemid in (51,442,455,6701,220179,220050) and valuenum > 0 and valuenum < 400 then 2 -- SysBP
    when itemid in (8368,8440,8441,8555,220180,220051) and valuenum > 0 and valuenum < 300 then 3 -- DiasBP
    when itemid in (456,52,6702,443,220052,220181,225312) and valuenum > 0 and valuenum < 300 then 4 -- MeanBP
    when itemid in (615,618,220210,224690) and valuenum > 0 and valuenum < 70 then 5 -- RespRate
    when itemid in (223761,678) and valuenum > 70 and valuenum < 120  then 6 -- TempF, converted to degC in valuenum call
    when itemid in (223762,676) and valuenum > 10 and valuenum < 50  then 6 -- TempC
    when itemid in (646,220277) and valuenum > 0 and valuenum <= 100 then 7 -- SpO2
    when itemid in (807,811,1529,3745,3744,225664,220621,226537) and valuenum > 0 then 8 -- Glucose
    else null end as vitalid, -- convert F to C

	case when itemid in (223761,678) then (valuenum-32)/1.8 else valuenum end as valuenum

  from admissions ad
  left join chartevents ce
  on ad.hadm_id = ce.hadm_id
  and ce.charttime between ad.admittime AND ad.dischtime
  -- exclude rows marked as error
  and (ce.error IS NULL or ce.error = 0)
  where ad.hadm_id in ({considered_hadm_id_str})
  AND ce.itemid in (
  -- HEART RATE
  211, --"Heart Rate"
  220045, --"Heart Rate"

  -- Systolic/diastolic

  51, --	Arterial BP [Systolic]
  442, --	Manual BP [Systolic]
  455, --	NBP [Systolic]
  6701, --	Arterial BP #2 [Systolic]
  220179, --	Non Invasive Blood Pressure systolic
  220050, --	Arterial Blood Pressure systolic

  8368, --	Arterial BP [Diastolic]
  8440, --	Manual BP [Diastolic]
  8441, --	NBP [Diastolic]
  8555, --	Arterial BP #2 [Diastolic]
  220180, --	Non Invasive Blood Pressure diastolic
  220051, --	Arterial Blood Pressure diastolic


  -- MEAN ARTERIAL PRESSURE
  456, --"NBP Mean"
  52, --"Arterial BP Mean"
  6702, --	Arterial BP Mean #2
  443, --	Manual BP Mean(calc)
  220052, --"Arterial Blood Pressure mean"
  220181, --"Non Invasive Blood Pressure mean"
  225312, --"ART BP mean"

  -- RESPIRATORY RATE
  618,--	Respiratory Rate
  615,--	Resp Rate (Total)
  220210,--	Respiratory Rate
  224690, --	Respiratory Rate (Total)


  -- SPO2, peripheral
  646, 220277,

  -- GLUCOSE, both lab and fingerstick
  807,--	Fingerstick Glucose
  811,--	Glucose (70-105)
  1529,--	Glucose
  3745,--	BloodGlucose
  3744,--	Blood Glucose
  225664,--	Glucose finger stick
  220621,--	Glucose (serum)
  226537,--	Glucose (whole blood)

  -- TEMPERATURE
  223762, -- "Temperature Celsius"
  676,	-- "Temperature C"
  223761, -- "Temperature Fahrenheit"
  678 --	"Temperature F"
	)
) as pvt
group by pvt.subject_id, pvt.hadm_id, pvt.charttime
order by pvt.subject_id, pvt.hadm_id, pvt.charttime
"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "vital.csv"),index=False,sep=',')
df.info()

## Laboratory FROM Chartevents


In [ ]:
query = f"""
 SELECT ad.subject_id, ad.hadm_id, EXTRACT(EPOCH FROM le.charttime - ad.admittime) as charttime
  -- here we assign labels to ITEMIDs
  -- this also fuses together multiple ITEMIDs containing the same data
  , CASE
        WHEN itemid = 50868 THEN 'ANION GAP'
        WHEN itemid = 50862 THEN 'ALBUMIN'
        WHEN itemid = 51144 THEN 'BANDS'
        WHEN itemid = 50882 THEN 'BICARBONATE'
        WHEN itemid = 50885 THEN 'BILIRUBIN'
        WHEN itemid = 50912 THEN 'CREATININE'
        WHEN itemid = 50806 THEN 'CHLORIDE'
        WHEN itemid = 50902 THEN 'CHLORIDE'
        WHEN itemid = 50809 THEN 'GLUCOSE'
        WHEN itemid = 50931 THEN 'GLUCOSE'
        WHEN itemid = 50810 THEN 'HEMATOCRIT'
        WHEN itemid = 51221 THEN 'HEMATOCRIT'
        WHEN itemid = 50811 THEN 'HEMOGLOBIN'
        WHEN itemid = 51222 THEN 'HEMOGLOBIN'
        WHEN itemid = 50813 THEN 'LACTATE'
        WHEN itemid = 51265 THEN 'PLATELET'
        WHEN itemid = 50822 THEN 'POTASSIUM'
        WHEN itemid = 50971 THEN 'POTASSIUM'
        WHEN itemid = 51275 THEN 'PTT'
        WHEN itemid = 51237 THEN 'INR'
        WHEN itemid = 51274 THEN 'PT'
        WHEN itemid = 50824 THEN 'SODIUM'
        WHEN itemid = 50983 THEN 'SODIUM'
        WHEN itemid = 51006 THEN 'BUN'
        WHEN itemid = 51300 THEN 'WBC'
        WHEN itemid = 51301 THEN 'WBC'
      ELSE null
    END as label
  , -- add in some sanity checks on the values
  -- the WHERE clause below requires all valuenum to be > 0, so these are only upper limit checks
    CASE
      WHEN itemid = 50862 AND valuenum >    10 THEN null -- g/dL 'ALBUMIN'
      WHEN itemid = 50868 AND valuenum > 10000 THEN null -- mEq/L 'ANION GAP'
      WHEN itemid = 51144 AND valuenum <     0 THEN null -- immature band forms, %
      WHEN itemid = 51144 AND valuenum >   100 THEN null -- immature band forms, %
      WHEN itemid = 50882 AND valuenum > 10000 THEN null -- mEq/L 'BICARBONATE'
      WHEN itemid = 50885 AND valuenum >   150 THEN null -- mg/dL 'BILIRUBIN'
      WHEN itemid = 50806 AND valuenum > 10000 THEN null -- mEq/L 'CHLORIDE'
      WHEN itemid = 50902 AND valuenum > 10000 THEN null -- mEq/L 'CHLORIDE'
      WHEN itemid = 50912 AND valuenum >   150 THEN null -- mg/dL 'CREATININE'
      WHEN itemid = 50809 AND valuenum > 10000 THEN null -- mg/dL 'GLUCOSE'
      WHEN itemid = 50931 AND valuenum > 10000 THEN null -- mg/dL 'GLUCOSE'
      WHEN itemid = 50810 AND valuenum >   100 THEN null -- % 'HEMATOCRIT'
      WHEN itemid = 51221 AND valuenum >   100 THEN null -- % 'HEMATOCRIT'
      WHEN itemid = 50811 AND valuenum >    50 THEN null -- g/dL 'HEMOGLOBIN'
      WHEN itemid = 51222 AND valuenum >    50 THEN null -- g/dL 'HEMOGLOBIN'
      WHEN itemid = 50813 AND valuenum >    50 THEN null -- mmol/L 'LACTATE'
      WHEN itemid = 51265 AND valuenum > 10000 THEN null -- K/uL 'PLATELET'
      WHEN itemid = 50822 AND valuenum >    30 THEN null -- mEq/L 'POTASSIUM'
      WHEN itemid = 50971 AND valuenum >    30 THEN null -- mEq/L 'POTASSIUM'
      WHEN itemid = 51275 AND valuenum >   150 THEN null -- sec 'PTT'
      WHEN itemid = 51237 AND valuenum >    50 THEN null -- 'INR'
      WHEN itemid = 51274 AND valuenum >   150 THEN null -- sec 'PT'
      WHEN itemid = 50824 AND valuenum >   200 THEN null -- mEq/L == mmol/L 'SODIUM'
      WHEN itemid = 50983 AND valuenum >   200 THEN null -- mEq/L == mmol/L 'SODIUM'
      WHEN itemid = 51006 AND valuenum >   300 THEN null -- 'BUN'
      WHEN itemid = 51300 AND valuenum >  1000 THEN null -- 'WBC'
      WHEN itemid = 51301 AND valuenum >  1000 THEN null -- 'WBC'
    ELSE le.valuenum
    END as valuenum

 	FROM admissions ad, labevents le
    WHERE le.hadm_id = ad.hadm_id
        AND ad.hadm_id in ({considered_hadm_id_str})
		AND le.charttime BETWEEN (ad.admittime) AND (ad.dischtime)
		AND le.ITEMID in
		(
		  -- comment is: LABEL | CATEGORY | FLUID | NUMBER OF ROWS IN LABEVENTS
		  50868, -- ANION GAP | CHEMISTRY | BLOOD | 769895
		  50862, -- ALBUMIN | CHEMISTRY | BLOOD | 146697
		  51144, -- BANDS - hematology
		  50882, -- BICARBONATE | CHEMISTRY | BLOOD | 780733
		  50885, -- BILIRUBIN, TOTAL | CHEMISTRY | BLOOD | 238277
		  50912, -- CREATININE | CHEMISTRY | BLOOD | 797476
		  50902, -- CHLORIDE | CHEMISTRY | BLOOD | 795568
		  50806, -- CHLORIDE, WHOLE BLOOD | BLOOD GAS | BLOOD | 48187
		  50931, -- GLUCOSE | CHEMISTRY | BLOOD | 748981
		  50809, -- GLUCOSE | BLOOD GAS | BLOOD | 196734
		  51221, -- HEMATOCRIT | HEMATOLOGY | BLOOD | 881846
		  50810, -- HEMATOCRIT, CALCULATED | BLOOD GAS | BLOOD | 89715
		  51222, -- HEMOGLOBIN | HEMATOLOGY | BLOOD | 752523
		  50811, -- HEMOGLOBIN | BLOOD GAS | BLOOD | 89712
		  50813, -- LACTATE | BLOOD GAS | BLOOD | 187124
		  51265, -- PLATELET COUNT | HEMATOLOGY | BLOOD | 778444
		  50971, -- POTASSIUM | CHEMISTRY | BLOOD | 845825
		  50822, -- POTASSIUM, WHOLE BLOOD | BLOOD GAS | BLOOD | 192946
		  51275, -- PTT | HEMATOLOGY | BLOOD | 474937
		  51237, -- INR(PT) | HEMATOLOGY | BLOOD | 471183
		  51274, -- PT | HEMATOLOGY | BLOOD | 469090
		  50983, -- SODIUM | CHEMISTRY | BLOOD | 808489
		  50824, -- SODIUM, WHOLE BLOOD | BLOOD GAS | BLOOD | 71503
		  51006, -- UREA NITROGEN | CHEMISTRY | BLOOD | 791925
		  51301, -- WHITE BLOOD CELLS | HEMATOLOGY | BLOOD | 753301
		  51300  -- WBC COUNT | HEMATOLOGY | BLOOD | 2371
		)
		AND valuenum IS NOT null AND valuenum > 0 -- lab values cannot be 0 AND cannot be negative
	ORDER BY ad.subject_id, ad.hadm_id;"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "labs.csv"),index=False,sep=',')
df.info()

## Extract graph building information

ICD code and drug information

In [ ]:
query = f"""
SELECT hadm_id, icd9_code, seq_num
FROM diagnoses_icd 
WHERE icd9_code NOT LIKE 'V%'
AND icd9_code NOT LIKE 'E%'
AND hadm_id in ({considered_hadm_id_str})
"""

df = pd.read_sql_query(query,conn, dtype={'icd9_code': str})
df.to_csv(os.path.join(export_dir, "diag.csv"),index=False, sep=',')
df.info()

In [ ]:
query = f"""
Select p.hadm_id, p.drug, p.dose_val_rx, p.dose_unit_rx, d.icd9_code
From (
	SELECT hadm_id, drug, dose_val_rx, dose_unit_rx
	FROM prescriptions
	WHERE hadm_id in ({considered_hadm_id_str})
	) as p
LEFT JOIN diagnoses_icd d
ON p.hadm_id = d.hadm_id
"""

df = pd.read_sql_query(query,conn)
df.to_csv(os.path.join(export_dir, "drug.csv"),index=False, sep=',')
df.info()